In [3]:

import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("청년정책_최종.csv")


In [4]:
house_df = df[df['정책대분류명'].isin(['주거'])]

In [5]:
print(f"주거 관련 정책 수: {len(house_df)}")
print(house_df['정책대분류명'].value_counts())

주거 관련 정책 수: 271
정책대분류명
주거    271
Name: count, dtype: int64


In [6]:
# 2. 지역 그룹 분류 함수 정의
def region_label(zip_text):
    if pd.isnull(zip_text):
        return '기타'
    if '서울' in zip_text:
        return '서울'
    elif '경기' in zip_text:
        return '경기'
    else:
        return '지방'

In [7]:
# 3. '정책거주지역코드' 기준으로 지역 그룹 파생
house_df['지역그룹'] = house_df['정책거주지역코드'].apply(region_label)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_25680\336814964.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  house_df['지역그룹'] = house_df['정책거주지역코드'].apply(region_label)


In [8]:
# 4. 각 지역 그룹별 인기 정책 조회수 상위 5개 추출
popular_by_region = house_df.groupby('지역그룹').apply(
    lambda x: x.sort_values(by='조회수', ascending=False).head(5)
).reset_index(drop=True)


C:\Users\Playdata\AppData\Local\Temp\ipykernel_25680\1900110152.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  popular_by_region = house_df.groupby('지역그룹').apply(


In [9]:
# 5. 결과 확인
print("✅ 지역별 인기 정책 (조회수 상위 5개):")
print(popular_by_region[['지역그룹', '정책명', '조회수']])

✅ 지역별 인기 정책 (조회수 상위 5개):
   지역그룹                                          정책명   조회수
0    경기                              경기행복주택 예비입주자 모집   219
1    경기           신혼부부·청년 전·월세대출 보증금 이자 지원사업 추가모집 안내    97
2    경기   2024년 「경기 저소득층 전세금 대출보증 및 이자지원사업」신청자 모집 공고    81
3    경기                                  용인청년 창업지원주택    67
4    경기  수원청년 전용 역세권 임대주택 역세권 새빛 청년존(Zone) 2호 입주자 모집    60
5    서울                    청년안심주택 공급활성화(임차보증금 무이자지원)  3976
6    서울                                     청년 월세 지원  1091
7    서울                       청년 부동산 중개보수 및 이사비 지원사업  1063
8    서울                                청년안심주택 공급(매입)   485
9    서울                                 청년 매입임대주택 사업   441
10   지방                                   청년주택드림청약통장  3960
11   지방                          청년 신규 공무원 주거안정방안 마련  3950
12   지방                                  생활안정자금 융자사업  2238
13   지방                              관외 청년 거주정착 지원사업  1300
14   지방                           공공임대주택 임대차 보증금 지원   1020


In [10]:
# 경기도 주거 사업은 왜 인기가 없을까?
# 1. 경기도 주거 정책이 적나?
# 2. 금전 키워드가 안들어가 사람들의 관심이 적나?

In [11]:
# 2️⃣ 금전 키워드 포함 여부 (제목에)
money_words = ['지원', '보증', '수당', '금액', '현금', '대출', '월세', '임대']
def has_money_kw(text):
    return any(word in str(text) for word in money_words)

In [12]:
house_df['금전키워드'] = house_df['정책명'].apply(has_money_kw)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_25680\3242316881.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  house_df['금전키워드'] = house_df['정책명'].apply(has_money_kw)


In [13]:
# 3️⃣ 정책지원내용에 숫자(금액) 언급 여부
def has_amount(text):
    return bool(re.search(r'\d+', str(text)))

In [18]:
import re
house_df['금액언급'] = house_df['정책지원내용'].apply(has_amount)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_25680\1490552790.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  house_df['금액언급'] = house_df['정책지원내용'].apply(has_amount)


In [19]:
# ✅ 결과 테이블 정리
summary = house_df.groupby('지역그룹').agg({
    '정책명': 'count',
    '금전키워드': 'mean',
    '금액언급': 'mean'
}).rename(columns={
    '정책명': '정책수',
    '금전키워드': '금전키워드비율',
    '금액언급': '금액언급비율'
}).round(3)

In [57]:
print("📊 서울/경기/지방 주거정책 비교 (정책수, 금전키워드, 금액언급):")
print(summary)

📊 서울/경기/지방 주거정책 비교 (정책수, 금전키워드, 금액언급):
      정책수  금전키워드비율  금액언급비율
지역그룹                      
경기     45    0.822   0.933
서울     30    0.700   0.933
지방    196    0.740   0.934


In [20]:
# 경기도 정책은 좋다 → 그런데 사람들이 모르고 있다 → 추천 시스템
# 경기 지역 유저에게 경기 주거 정책 우선 노출!!

In [ ]:
# 지역별로 주거 중에서도 가장 관심이 높은 정책은 뭘까?

In [64]:
from collections import Counter
from itertools import chain
import re

In [69]:
# 한글 단어 추출 함수
def tokenize_korean(text):
    return re.findall(r'[가-힣]{2,}', str(text))


In [70]:
# 지역 키워드 매칭용 딕셔너리
region_keywords = {
    '서울': '서울',
    '경기': '경기',
    '지방': '서울|경기'  # 지방은 서울, 경기 아닌 나머지
}

In [72]:
# 지역별 처리
for region, keyword in region_keywords.items():
    if region != '지방':
        region_df = df[(df['정책대분류명'] == '주거') & df['정책거주지역코드'].str.contains(keyword, na=False)]
    else:
        region_df = df[(df['정책대분류명'] == '주거') & ~df['정책거주지역코드'].str.contains(keyword, na=False)]

    # 상위 50개 조회수 기준 정책 추출
    top_region_df = region_df.sort_values(by='조회수', ascending=False).head(50)
    
    # 제목에서 키워드 추출
    words = list(chain.from_iterable(top_region_df['정책지원내용'].apply(tokenize_korean)))
    word_counts = Counter(words)
    top_keywords = pd.Series(word_counts).sort_values(ascending=False).head(20)
    
    print(f"\n✅ [{region}] 지역 상위 정책 제목 키워드 TOP 20")
    print(top_keywords)


✅ [서울] 지역 상위 정책 제목 키워드 TOP 20
사업       32
만원       26
지원       25
청년       23
최대       20
신혼부부     16
이하       13
이내       13
월세       12
경우        9
모집        8
서울특별시     8
백만원       8
역세권       8
서울시       8
소득        8
임차보증금     7
기준        7
조례        7
대상        7
dtype: int64

✅ [경기] 지역 상위 정책 제목 키워드 TOP 20
최대        46
지원        42
만원        34
경기행복주택    20
지원내용      19
경기        16
이내        16
지급        13
월세        12
주택        12
개월        10
신혼부부      10
청년         8
가능         8
번길         7
신청         7
최장         7
경우         6
매년         6
임차료        6
dtype: int64

✅ [지방] 지역 상위 정책 제목 키워드 TOP 20
만원      50
지원      45
최대      40
이하      29
기준      23
경우      22
청년      22
지급      18
억원      18
개월      18
이내      18
월세      14
또는      14
주택      13
가능      12
지원내용    12
지원대상    12
이상      12
신청      11
최장      10
dtype: int64


* 지역별 인기 정책 <br>
경기도가 서울/지방에 비해 현저히 조회수가 낮다<br>
금전을 지원안하는 것도 아니다 -> 홍보가 안됨 -> 경기 지역 유저는 경기 주거 정책 우선 추천

* 전 지역에서 금전적 혜택이 가장 중요한 요소이다<br>
하지만 서울은 돈 뿐만 아니라 위치까지 중요!<br>
서울 사용자 추천 시에는 위치 관련 피처 추가 가능(역세권, 교통)

- 전체적으로 금전적 요소가 있냐 없냐를 신경 많이 씀 - 금전적 요소 유무 피처 추가 가능
- 주거 정책에서 서울이면 위치기반 키워드 유무 피처 추가 가능